In [1]:
import nltk
nltk.download('words')
from nltk.corpus import words

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


In [2]:
all_words = set(w.lower() for w in words.words())

In [3]:
len(all_words)

234377

In [4]:
import heapq
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import matplotlib.pyplot as plt

In [5]:
def refined_heuristic(word1, word2):

    return sum(c1 != c2 for c1, c2 in zip(word1, word2))

In [6]:
def get_neighbors(word, word_set):
    neighbors = []
    for i in range(len(word)):
        for c in 'abcdefghijklmnopqrstuvwxyz':
            if c != word[i]:
                new_word = word[:i] + c + word[i+1:]
                if new_word in word_set and new_word != word:
                    neighbors.append(new_word)
    return neighbors

In [11]:
import itertools
import heapq

def a_star_trace(start, goal, word_set):
    open_set = []
    counter = itertools.count()#Initializing the tie-breaker

    heapq.heappush(open_set, (refined_heuristic(start, goal), next(counter), start, [start]))
    visited = set()
    steps = []

    while open_set:
        _, _, current, path = heapq.heappop(open_set)
        visited.add(current)

        h = refined_heuristic(current, goal)

        step_log = {
            "current": current,
            "path": path.copy(),
            "neighbors": get_neighbors(current, word_set),
            "visited": visited.copy(),
            "h": h
        }

        if not steps or steps[-1]["current"] != current:#skipping consecutive duplicate steps
            steps.append(step_log)

        if current == goal:
            return path, steps

        neighbors = get_neighbors(current, word_set)


        for neighbor in neighbors:
            if neighbor not in visited:
                f_cost = refined_heuristic(neighbor, goal) + len(path)
                heapq.heappush(open_set, (
                    f_cost,
                    next(counter),
                    neighbor,
                    path + [neighbor]
                ))

    return None, steps

In [8]:
def draw_all_steps(steps, goal, max_steps=None):
    import matplotlib.pyplot as plt
    import networkx as nx

    n_steps = len(steps)
    max_steps_per_row = 5
    n_rows = (n_steps + max_steps_per_row - 1) // max_steps_per_row

    fig_width = max(5 * max_steps_per_row, 15)
    fig_height = 5 * n_rows

    fig, axes = plt.subplots(n_rows, max_steps_per_row, figsize=(fig_width, fig_height))
    axes = axes.flatten()

    valid_words = [step['current'] for step in steps]
    for i in range(n_steps):
        step = steps[i]
        if step['current'] not in valid_words:
            continue

        G = nx.Graph()
        center = step["current"]
        neighbors = step["neighbors"]
        path = step["path"]

        G.add_node(center)
        for n in neighbors:
            G.add_node(n)
            G.add_edge(center, n)

        color_map = []
        for node in G.nodes():
            if node == center:
                color_map.append("orange")
            elif node == goal:
                color_map.append("green")
            elif node in path:
                color_map.append("gold")
            elif node in step["visited"]:
                color_map.append("gray")
            else:
                color_map.append("skyblue")


        pos = nx.spring_layout(G, seed=42, k=0.5)
        ax = axes[i]
        ax.set_title(f"Step {i+1}\n{center}", fontsize=10)
        nx.draw(G, pos, with_labels=True, node_color=color_map, node_size=1000, font_size=8, ax=ax)

    for j in range(n_steps, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


    print("Heuristicvalues:")
    for i in range(n_steps):
        step = steps[i]
        print(f"Step {i+1}: {step['current']} → h(n) = {step['h']}")

In [9]:
start_input = widgets.Text(description="Start Word:")
goal_input = widgets.Text(description="Goal Word:")
run_button = widgets.Button(description="Run")
output_area = widgets.Output()

In [16]:
def on_run_click(b):
    with output_area:
        clear_output()
        start = start_input.value.lower().strip()
        goal = goal_input.value.lower().strip()

        if len(start) != len(goal):
            print("Start and goal words must be the same length.")
            return

        if start == goal:
            print("Start and goal words must be different.")
            return

        word_set = {w for w in all_words if len(w) == len(start)}
        if start not in word_set or goal not in word_set:
            print("not found in dictionary.")
            return

        path, steps = a_star_trace(start, goal, word_set)
        if path:
          a=[]
          n_steps = len(steps)
          for i in range(n_steps):
            step = steps[i]
            a.append(step['current'])

          print("Word Ladder Found:", " → ".join(a))
        else:
            print("No path found.")

        draw_all_steps(steps, goal, max_steps=None)

run_button.on_click(on_run_click)

# Display UI
display(start_input, goal_input, run_button, output_area)


Text(value='Above', description='Start Word:')

Text(value='place', description='Goal Word:')

Button(description='Run', style=ButtonStyle())

Output()